In [0]:
-- Compare silver with gold to detect changes, expire old active records
MERGE INTO retail_lakehouse.gold.dim_customer tgt
USING retail_lakehouse.silver.customers src
ON tgt.CustomerID = src.CustomerID
AND tgt.IsActive = TRUE
WHEN MATCHED
AND (
    tgt.City <> src.City
    OR tgt.Address <> src.Address
    OR tgt.Email <> src.Email
)
THEN UPDATE SET
    tgt.EndDate = CURRENT_DATE,
    tgt.IsActive = FALSE;

In [0]:
-- Insert new version for customers without active records in gold
INSERT INTO retail_lakehouse.gold.dim_customer
(
    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    StartDate,
    EndDate,
    IsActive
)

SELECT
    src.CustomerID,
    src.CustomerName,
    src.Email,
    src.City,
    src.Address,
    CURRENT_DATE,
    DATE '9999-12-31',
    TRUE
FROM retail_lakehouse.silver.customers src
LEFT JOIN retail_lakehouse.gold.dim_customer tgt
ON src.CustomerID = tgt.CustomerID
AND tgt.IsActive = TRUE
WHERE tgt.CustomerID IS NULL;

SCD Type 2 Validation Checks

In [0]:
%sql
-- Should return 0 rows - each customer should have exactly 1 active record
SELECT
    CustomerID,
    COUNT(*) AS active_count
FROM retail_lakehouse.gold.dim_customer
WHERE IsActive = TRUE
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:
%sql
-- View all versions (current and historical) for a customer
SELECT
    CustomerSK,
    CustomerID,
    CustomerName,
    City,
    Address,
    Email,
    StartDate,
    EndDate,
    IsActive
FROM retail_lakehouse.gold.dim_customer
WHERE CustomerID = 1
ORDER BY StartDate DESC;

In [0]:
%sql
-- Summary statistics: unique customers vs total versions
SELECT
    COUNT(DISTINCT CustomerID) AS unique_customers,
    COUNT(*) AS total_versions,
    SUM(CASE WHEN IsActive = TRUE THEN 1 ELSE 0 END) AS active_versions,
    SUM(CASE WHEN IsActive = FALSE THEN 1 ELSE 0 END) AS historical_versions
FROM retail_lakehouse.gold.dim_customer;